In [16]:
import sys
import re
import os
import ipywidgets as widgets
from ipywidgets import interact
import h5py
import numpy as np
import json

import healpy as hp

sys.path.append(os.path.join(os.getcwd(), "scripts"))
from utils.plots import *
from utils import Config

In [17]:
alm = None
fnl = None

# Get a list of all files in the 'data/alms' directory
files = os.listdir('data/alms')

# Create a dropdown widget with the files as options
dropdown = widgets.Dropdown(options=files)

simulation_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=99999,  # Adjust this value based on the number of simulations
    step=1,
    description='Simulation:',
    continuous_update=False
)

# Create a button selector for the second dimension (polarization)
polarization_buttons = widgets.ToggleButtons(
    options=['T', 'E', 'B'],
    description='Polarization:',
    button_style='',  # 'success', 'info', 'warning', 'danger' or ''
    tooltips=['Temperature', 'Electric', 'Magnetic'],
)

# Define a function to update the file based on the selected simulation and polarization
def update_file(file, simulation, polarization):
    global alm, fnl
    # Load the file
    file_path = 'data/alms/{}'.format(file)  # Adjust this path based on your file structure
    file = h5py.File(file_path, 'r')
    print(file.keys())

    simulation_slider.max = file['alm'].shape[0] - 1

    # Update the options of the polarization_buttons based on alm.shape[1]
    valid_options = ['T', 'E', 'B'][:file['alm'].shape[1]]
    polarization_buttons.options = valid_options
    polarization_index = valid_options.index(polarization)

    # Assuming 'alm' and 'fnl' are datasets in the file
    alm = file['alm'][simulation, polarization_index].astype(np.complex128)
    fnl = file['fnl'][simulation, polarization_index]

    # Calculate and print various statistics for 'alm'
    print("Statistics for 'alm':")
    print("Mean:", np.mean(alm))
    print("Median:", np.median(alm))
    print("Standard deviation:", np.std(alm))
    print("Min:", np.min(alm))
    print("Max:", np.max(alm))
    print("Range:", np.ptp(alm))  # Peak-to-peak (max - min)
    print("Variance:", np.var(alm))
    print("fnl:", fnl)

    plot_cl_alm(alm)

    nside = int(re.search(r'_n(\d+)_', file_path).group(1))
    hp.mollview(hp.sphtfunc.alm2map(alm, nside), title='Alm')

# Call update_file whenever simulation_slider.value or polarization_buttons.value changes
widgets.interactive(update_file, file=dropdown, simulation=simulation_slider, polarization=polarization_buttons)

interactive(children=(Dropdown(description='file', options=('l500_n128_l_T_100000.alms.hdf5', 'l2000_n2048_l_T…

In [18]:
import tensorflow as tf
from tensorflow.data import AUTOTUNE

# testing a possible bug
dataset = tf.data.Dataset.range(1000)
train_size = int(1000 * .8)
val_size = test_size = int(1000 * .1)

def get_dataset(start, step):
    data = dataset.skip(start).take(step)
    # data = data.apply(assert_cardinality(step))
    data = data.cache()
    data = data.shuffle(1000, reshuffle_each_iteration=True)
    data = data.batch(
        8,
        drop_remainder=True,
        deterministic=False,
        num_parallel_calls=AUTOTUNE
    )
    return data.prefetch(AUTOTUNE)

train_ds = get_dataset(0, train_size)
test_ds = get_dataset(train_size, test_size)
val_ds = get_dataset(train_size + test_size, val_size)

def check(a, b, c):
    a = set(tuple(x) for x in a.as_numpy_iterator())
    b = set(tuple(x) for x in b.as_numpy_iterator())
    c = set(tuple(x) for x in c.as_numpy_iterator())

    ab = a.intersection(b)
    bc = b.intersection(c)
    ac = a.intersection(c)

    print('intersections', ab, bc, ac)

check(train_ds, test_ds, val_ds)

intersections set() set() set()


In [31]:
import tensorflow as tf
from tensorflow.data import AUTOTUNE

def generator():
    for i in np.arange(1000):
        yield (i,)

# testing a possible bug
dataset = tf.data.Dataset.from_generator(
    generator,
    output_signature=(
                tf.TensorSpec(shape=(), dtype=tf.int64),
            )
)
train_size = int(1000 * .8)
val_size = test_size = int(1000 * .1)

def get_dataset(start, step):
    data = dataset.skip(start).take(step)
    # data = data.apply(assert_cardinality(step))
    data = data.cache()
    data = data.shuffle(1000, reshuffle_each_iteration=True)
    data = data.batch(
        8,
        drop_remainder=True,
        deterministic=False,
        num_parallel_calls=AUTOTUNE
    )
    return data.prefetch(AUTOTUNE)

train_ds = get_dataset(0, train_size)
test_ds = get_dataset(train_size, test_size)
val_ds = get_dataset(train_size + test_size, val_size)

def check(a, b, c):
    a = list(tuple(x) for x in a.as_numpy_iterator())
    b = list(tuple(x) for x in b.as_numpy_iterator())
    c = list(tuple(x) for x in c.as_numpy_iterator())

    ab = np.intersect1d(a, b)
    bc = np.intersect1d(b,c)
    ac = np.intersect1d(a,c)

    print('intersections', ab, bc, ac)

check(train_ds, test_ds, val_ds)

intersections [] [] []
